In [20]:
!pip install undetected-chromedriver

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47214 sha256=873728d8ea87ef80fe92d5587c7ca620da6c1f4148d7e6376e224b692be9421b
  Stored in directory: c:\users\hp5cd\appdata\local\pip\cache\wheels\c4\f1\aa\9de6cf276210554d91e9c0526864563e850a428c5e76da4914
Successfully built undetected-chromedriver



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# Setup driver
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

In [30]:

base_url = "https://www.flipkart.com/titan-mechanical-automatic-grey-dial-black-leather-strap-analog-watch-men/product-reviews/itm3799ec276b122?pid=WATH47UHUJVWDBY3&lid=LSTWATH47UHUJVWDBY3IWMGWD&marketplace=FLIPKART"

In [31]:

all_reviews = []

for page in range(1, 4):
    url = base_url + f"&page={page}"
    print("Scraping:", url)

    driver.get(url)

    # Close login popup
    try:
        WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, '//button[contains(text(),"✕")]'))
        ).click()
    except:
        pass

    # ✅ Wait for ratings (means reviews loaded)
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, '//div[contains(@class,"_3LWZlK")]'))
        )
    except:
        print("⚠️ Reviews not loaded (blocked or slow)")
        continue

    time.sleep(2)

    # ✅ Real review containers
    reviews = driver.find_elements(By.XPATH, '//div[contains(@class,"_27M-vq")]')

    print("Found reviews:", len(reviews))

    for r in reviews:
        try:
            name = r.find_element(By.XPATH, './/p[contains(@class,"_2sc7ZR")]').text
            rating = r.find_element(By.XPATH, './/div[contains(@class,"_3LWZlK")]').text
            comment = r.find_element(By.XPATH, './/div[contains(@class,"t-ZTKy")]').text

            all_reviews.append({
                "Customer Name": name,
                "Rating": rating,
                "Comment": comment
            })

        except:
            continue

Scraping: https://www.flipkart.com/titan-mechanical-automatic-grey-dial-black-leather-strap-analog-watch-men/product-reviews/itm3799ec276b122?pid=WATH47UHUJVWDBY3&lid=LSTWATH47UHUJVWDBY3IWMGWD&marketplace=FLIPKART&page=1
⚠️ Reviews not loaded (blocked or slow)
Scraping: https://www.flipkart.com/titan-mechanical-automatic-grey-dial-black-leather-strap-analog-watch-men/product-reviews/itm3799ec276b122?pid=WATH47UHUJVWDBY3&lid=LSTWATH47UHUJVWDBY3IWMGWD&marketplace=FLIPKART&page=2
⚠️ Reviews not loaded (blocked or slow)
Scraping: https://www.flipkart.com/titan-mechanical-automatic-grey-dial-black-leather-strap-analog-watch-men/product-reviews/itm3799ec276b122?pid=WATH47UHUJVWDBY3&lid=LSTWATH47UHUJVWDBY3IWMGWD&marketplace=FLIPKART&page=3
⚠️ Reviews not loaded (blocked or slow)


In [19]:
driver.quit()

df = pd.DataFrame(all_reviews)
print("Total Reviews:", len(df))

df.to_csv("flipkart_reviews.csv", index=False)
print("Saved successfully")

Total Reviews: 0
File saved successfully
